# A first hidden Markov model

We ask one question. How can a sequence of noisy observations tell us about a sequence of hidden states?

A hidden Markov model (HMM) has three parts.

1. **Hidden state.** $Z_t\in\{0,1\}$ is the state at position $t$.
2. **Transition.** $P(Z_t\neq Z_{t-1})=s$ controls how often the state changes.
3. **Emission.** $P(X_t\neq Z_t)=e$ controls how often the observation differs from the state.

The forward--backward algorithm uses **dynamic programming** to sum over all $2^T$ possible hidden paths without listing them. It computes $P(Z_t=1\mid X_1,\ldots,X_T)$ at every position.

## One fixed sequence

We use one simulated hidden path and four deliberately flipped observations. The hidden path is shown because this is a teaching example. In real data, it is unknown.

In [1]:
library(hpgR)

hidden_state <- c(rep(0, 8), rep(1, 9), rep(0, 7), rep(1, 8))
observed <- hidden_state
flip_positions <- c(4, 12, 20, 27)
observed[flip_positions] <- 1 - observed[flip_positions]
position <- seq_along(observed)


sequence_data <- data.frame(
  position, hidden_state, observed,
  flipped = position %in% flip_positions
)
sequence_data[sequence_data$flipped, ]


,position,hidden_state,observed,flipped
,<int>,<dbl>,<dbl>,<lgl>
4,4,0,1,TRUE
12,12,1,0,TRUE
20,20,0,1,TRUE
27,27,1,0,TRUE


## Transition probability controls persistence

Move the slider for $s$. A small value says neighboring positions probably share a state, so the posterior smooths over isolated observations. A larger value permits more frequent state changes, so the posterior follows local observations more closely.

In [2]:
switch_grid <- c(0.01, 0.03, 0.05, 0.10, 0.20, 0.35, 0.50)
switch_frames <- do.call(rbind, lapply(switch_grid, function(s) {
  data.frame(
    position = position,
    hidden_state = hidden_state,
    observed = observed,
    posterior = binaryHMM(observed, s, emission_error = 0.15),
    setting = sprintf("%.2f", s)
  )
}))
switch_frames$setting <- factor(
  switch_frames$setting, levels = sprintf("%.2f", switch_grid)
)

invisible(switch_frames)


```{anywidget} ./widgets/hmm-basics.mjs
{"parameter": "switch"}
```

A posterior probability is not a decoded path. It measures uncertainty at each position. Values near $1/2$ mean that the two hidden states remain difficult to distinguish.

## Emission error controls how much the observations are trusted

Now hold $s=0.08$ fixed and move the slider for $e$. When $e$ is small, an observation strongly favors the matching state. As $e$ approaches $1/2$, the observations carry less information and the posterior is driven mainly by state persistence.

In [3]:
error_grid <- c(0.01, 0.05, 0.10, 0.20, 0.35, 0.49)
error_frames <- do.call(rbind, lapply(error_grid, function(e) {
  data.frame(
    position = position,
    hidden_state = hidden_state,
    observed = observed,
    posterior = binaryHMM(
      observed, switch_probability = 0.08, emission_error = e
    ),
    setting = sprintf("%.2f", e)
  )
}))
error_frames$setting <- factor(
  error_frames$setting, levels = sprintf("%.2f", error_grid)
)

invisible(error_frames)


```{anywidget} ./widgets/hmm-basics.mjs
{"parameter": "error"}
```

## What to remember

1. The **transition probability** describes how states persist or change along a sequence.
2. The **emission probability** describes how a hidden state produces an observation.
3. Forward--backward uses **dynamic programming** to sum over all possible hidden paths and return a posterior probability at each position.

The Li–Stephens model uses the same structure. The hidden state is the copied reference haplotype, a transition changes the copying source, and an emission mismatch represents imperfect copying.